In [1]:
# Install required libraries
!pip install -q -U transformers accelerate bitsandbytes datasets pandas langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 34.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 74.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configuration
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert DEVICE == "cuda", "GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU"

print(f"Loading tokenizer and model: {MODEL_ID} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading tokenizer and model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B on cuda...


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
pip install langdetect

In [4]:
import re
import os
import time
import pandas as pd
import langdetect

def extract_ground_truth(solution_str):
    """Extracts the final numerical answer from GSM8K ground truth string."""
    match = re.search(r"####\s*(-?[\d,]+(?:\.\d+)?)", solution_str)
    return match.group(1).replace(",", "").strip() if match else ""

def parse_numeric_answer(text):
    """Extracts the first standalone scalar number from generated response."""
    if not text:
        return ""
    cleaned = text.replace("$", "").replace(",", "")
    numbers = re.findall(r"-?\d+(?:\.\d+)?", cleaned)
    return numbers[0].strip() if numbers else ""

def calculate_target_language_ratio(text, target_lang_code="en"):
    """Calculates the percentage of sentences detected in the target language."""
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if not sentences:
        return 0.0
    target_count = 0
    for sent in sentences:
        try:
            if langdetect.detect(sent) == target_lang_code:
                target_count += 1
        except:
            pass
    return (target_count / len(sentences)) * 100

In [5]:
DATASET_PATH = "nvidia/GSM8K-Hi"
DATASET_SPLIT = "test"
OUTPUT_CSV = "gsm8k_hindi_results.csv"
LANGUAGE_NAME = "Hindi"
TARGET_LANG_CODE = "hi"
TARGET_BUDGETS = [50, 100, 200, 400, 800]

In [8]:
from datasets import load_dataset
print(f"Loading {LANGUAGE_NAME} dataset: {DATASET_PATH}...")
raw_dataset = load_dataset(DATASET_PATH, split=DATASET_SPLIT)
print(f"Dataset loaded! Total questions: {len(raw_dataset)}")

Loading Hindi dataset: nvidia/GSM8K-Hi...


README.md:   0%|          | 0.00/2.61k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.62M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded! Total questions: 1319


In [ ]:
# ---------------------------------------------------------
# 4. Check Resume Status
# ---------------------------------------------------------
completed_sample_ids = set()
if os.path.exists(OUTPUT_CSV):
    existing_df = pd.read_csv(OUTPUT_CSV)
    if "sample_id" in existing_df.columns:
        completed_sample_ids = set(existing_df["sample_id"].astype(str))
    print(f"🔄 Resuming existing run. Found {len(completed_sample_ids)} finished evaluations.")

print(f"🚀 Starting {LANGUAGE_NAME} evaluation loop for {len(raw_dataset)} questions across {len(TARGET_BUDGETS)} budgets...\n")

# ---------------------------------------------------------
# 5. Main Execution Loop
# ---------------------------------------------------------
start_time_total = time.time()

for idx, item in enumerate(raw_dataset):
    # Unique Question ID for Hindi
    question_id = f"GSM8K_HI_{idx+1}"
    question_text = item["question"]
    ground_truth = extract_ground_truth(item["answer"])

    messages = [{"role": "user", "content": question_text}]
    base_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(base_prompt, return_tensors="pt").to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    for budget in TARGET_BUDGETS:
        # Unique Sample ID for Hindi Run
        sample_id = f"{question_id}_b{budget}"

        if sample_id in completed_sample_ids:
            continue

        # --- STEP 1: Generate Thinking Up to Budget ---
        t0 = time.time()
        gen_output = model.generate(
            **inputs,
            max_new_tokens=budget,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.synchronize()
        step1_time = time.time() - t0

        raw_gen_text = tokenizer.decode(gen_output[0][prompt_len:], skip_special_tokens=False)

        if "</think>" in raw_gen_text:
            thinking_trace = raw_gen_text.split("</think>")[0].replace("<think>", "").strip()
            cutoff_status = "Finished_Thinking_Naturally"
        else:
            thinking_trace = raw_gen_text.replace("<think>", "").strip()
            cutoff_status = "Thinking_Cut_Off_By_Budget"

        thinking_tokens_used = len(tokenizer.encode(thinking_trace))
        lang_ratio = calculate_target_language_ratio(thinking_trace, TARGET_LANG_CODE)

        # --- STEP 2: Force Immediate Answer Extraction ---
        continuation_prompt = f"{base_prompt}<think>\n{thinking_trace}\n</think>\n\nFinal Answer: "
        forced_inputs = tokenizer(continuation_prompt, return_tensors="pt").to(DEVICE)
        forced_prompt_len = forced_inputs["input_ids"].shape[1]

        t0 = time.time()
        ans_output = model.generate(
            **forced_inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        torch.cuda.synchronize()
        step2_time = time.time() - t0

        raw_ans_text = tokenizer.decode(ans_output[0][forced_prompt_len:], skip_special_tokens=True).strip()
        parsed_answer = parse_numeric_answer(raw_ans_text)
        is_correct = (parsed_answer == ground_truth)

        # --- STEP 3: Record & Save Immediately ---
        record = {
            "question_id": question_id,
            "language": LANGUAGE_NAME,
            "model": MODEL_ID.split("/")[-1],
            "sample_id": sample_id,
            "budget": budget,
            "full_thinking_text": thinking_trace,
            "raw_answer_text": raw_ans_text,
            "parsed_answer": parsed_answer,
            "ground_truth": ground_truth,
            "correct": is_correct,
            "tokens_used": thinking_tokens_used,
            "step1_time_sec": round(step1_time, 2),
            "step2_time_sec": round(step2_time, 2),
            "target_lang_ratio_pct": round(lang_ratio, 1),
            "cutoff_status": cutoff_status
        }

        df_row = pd.DataFrame([record])
        df_row.to_csv(OUTPUT_CSV, mode="a", index=False, header=not os.path.exists(OUTPUT_CSV))
        completed_sample_ids.add(sample_id)

    # Progress output
    if (idx + 1) % 10 == 0 or (idx + 1) == len(raw_dataset):
        elapsed = time.time() - start_time_total
        print(f"Progress: [{idx + 1}/{len(raw_dataset)}] questions processed | Elapsed Time: {elapsed/60:.1f} mins")

print(f"\n✅ {LANGUAGE_NAME} benchmarking complete! All results saved to `{OUTPUT_CSV}`.")

🚀 Starting Hindi evaluation loop for 1319 questions across 5 budgets...

Progress: [10/1319] questions processed | Elapsed Time: 10.2 mins
